# 03 — Matrix Multiplication
## The Most Important Operation in ML

> **Prerequisites:** Notebooks 01 and 02  
> **Difficulty:** ⭐⭐⭐☆☆ Intermediate  
> **Running example:** Neural network forward pass

---

## Why is this the most important operation?

Every time a neural network makes a prediction, it does matrix multiplication. Every time you compute a linear regression, it's matrix multiplication. Image recognition, language translation, recommendation systems — all of them multiply matrices thousands of times per second.

> **If you understand matrix multiplication, you understand the computational core of all ML.**

---

## Quick reminder: what is it NOT?

`A * B` in NumPy = **element-wise multiply** (cell × cell, same shape required).  
`A @ B` in NumPy = **matrix multiply** (what we're learning now — different rule!)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

np.random.seed(42)

---
## 1. The Shape Rule — When Can You Multiply?

### Plain English
To multiply A (m×n) by B (n×p), the **inner numbers must match**.

```
    A        @     B       =     C
 (m × n)       (n × p)       (m × p)
        ↑     ↑
    these must match!

Result shape: (outer numbers) = (m × p)
```

### Memory trick: **(rows × cols) × (cols × rows)** — the shared dimension cancels.

### Why does this rule exist?
Each element of C is computed as: row i of A dotted with column j of B.
For that dot product to work, row i and column j must have the **same number of elements**.

In [ ]:
# ─────────────────────────────────────────────────────────
# THE SHAPE RULE
# ─────────────────────────────────────────────────────────

print("=== Shape rule examples ===")

# (2×3) @ (3×2) → (2×2) : inner dims both 3 ✓
A = np.array([[1., 2., 3.],
              [4., 5., 6.]])        # (2, 3)
B = np.array([[7., 8.],
              [9., 10.],
              [11., 12.]])          # (3, 2)
C = A @ B
print(f"A {A.shape} @ B {B.shape} → C {C.shape}")
print(f"Inner dims: {A.shape[1]} == {B.shape[0]} ✓")

# (5×4) @ (4×1) → (5×1): dataset × weights = predictions
X = np.array([[1500., 3., 10., 5.2],
              [2100., 4.,  5., 2.8],
              [1200., 2., 20., 8.0],
              [1800., 3.,  8., 4.5],
              [2500., 5.,  2., 1.0]])  # (5, 4)

w = np.array([[150.], [20000.], [-500.], [-8000.]])  # (4, 1) — one weight per feature
predictions = X @ w   # (5, 4) @ (4, 1) = (5, 1)
print(f"\nX {X.shape} @ w {w.shape} → predictions {predictions.shape}")
print(f"Predicted prices: {predictions.flatten()}")

# Show what fails
print("\n=== What fails ===")
try:
    bad = np.ones((3, 2)) @ np.ones((3, 4))  # inner: 2 ≠ 3 → error
except ValueError as e:
    print(f"(3×2) @ (3×4): Error! {e}")

---
## 2. How It Works: The Dot Product Formula

### Each element in the result is a dot product

Element **C[i, j]** = dot product of **row i** from A with **column j** from B.

```
C[i,j] = A[i,0]×B[0,j] + A[i,1]×B[1,j] + A[i,2]×B[2,j] + ...
        = Σₖ A[i,k] × B[k,j]
```

### Step-by-step example
```
A = [[1, 2],    B = [[5, 6],
     [3, 4]]         [7, 8]]

C[0,0] = row0(A) · col0(B) = [1,2] · [5,7] = 1×5 + 2×7 = 19
C[0,1] = row0(A) · col1(B) = [1,2] · [6,8] = 1×6 + 2×8 = 22
C[1,0] = row1(A) · col0(B) = [3,4] · [5,7] = 3×5 + 4×7 = 43
C[1,1] = row1(A) · col1(B) = [3,4] · [6,8] = 3×6 + 4×8 = 50
```

In [ ]:
# ─────────────────────────────────────────────────────────
# COMPUTING EACH ELEMENT MANUALLY
# ─────────────────────────────────────────────────────────

A = np.array([[1., 2.],
              [3., 4.]])
B = np.array([[5., 6.],
              [7., 8.]])

print("=== Manual computation of A @ B ===")
print(f"A:\n{A}")
print(f"B:\n{B}")
print()

# Compute each element manually to show what's happening
for i in range(A.shape[0]):
    for j in range(B.shape[1]):
        row = A[i, :]       # row i of A
        col = B[:, j]       # column j of B
        dot = np.dot(row, col)
        terms = " + ".join([f"{int(row[k])}×{int(col[k])}" for k in range(len(row))])
        print(f"C[{i},{j}] = row {i} of A · col {j} of B")
        print(f"        = {list(row.astype(int))} · {list(col.astype(int))}")
        print(f"        = {terms} = {int(dot)}")
        print()

print(f"Result C = A @ B:")
print(A @ B)
print(f"\nNumPy confirms: {(A @ B == np.array([[19,22],[43,50]])).all()}")

---
## 3. ML Meaning: A Neural Network Layer

### The big picture
A neural network layer does exactly this:
```
output = input @ weights + bias
  (m×n)   (m×n)  (n×p)   (p,)
```

- Each **row** of the input = one data sample (one house)
- Each **column** of the weights = one neuron's weights
- Each **element** of the output = how much one sample activates one neuron

This is how **32 samples can be processed simultaneously** in one matrix multiply — that's what makes GPUs fast for ML.

In [ ]:
# ─────────────────────────────────────────────────────────
# NEURAL NETWORK FORWARD PASS
# ─────────────────────────────────────────────────────────

print("=== Neural Network Forward Pass ===")
print()

# Input: 3 houses, each with 4 features
X_batch = np.array([
    [1500., 3., 10., 5.2],   # House 1
    [2100., 4.,  5., 2.8],   # House 2
    [1200., 2., 20., 8.0],   # House 3
])  # shape (3, 4)

# Layer 1 weights: 4 inputs → 5 hidden neurons
np.random.seed(0)
W1 = np.random.randn(4, 5) * 0.1   # shape (4, 5)
b1 = np.zeros(5)                    # shape (5,) — one bias per neuron

print(f"Input X:     shape {X_batch.shape}  (3 samples, 4 features)")
print(f"Weights W1:  shape {W1.shape}  (4 inputs → 5 neurons)")
print(f"Bias b1:     shape {b1.shape}")
print()

# Forward pass through layer 1
Z1 = X_batch @ W1 + b1   # (3,4) @ (4,5) + (5,) → (3,5)
print(f"Z1 = X @ W1 + b1")
print(f"Z1 shape: {Z1.shape}  (3 samples, 5 neuron outputs)")
print(f"Z1 (first 3 decimals):")
print(Z1.round(3))
print()
print(f"Z1[0] = House 1's activations: {Z1[0].round(3)}")
print(f"Z1[1] = House 2's activations: {Z1[1].round(3)}")
print(f"Z1[2] = House 3's activations: {Z1[2].round(3)}")

# Apply ReLU activation: max(0, x)
A1 = np.maximum(0, Z1)  # ReLU: negative values become 0
print(f"\nAfter ReLU activation A1:")
print(A1.round(3))
print("→ Negative values are zeroed out (neuron 'doesn't fire')")

# Layer 2: 5 hidden neurons → 1 output (price prediction)
W2 = np.random.randn(5, 1) * 0.1
b2 = np.array([50000.])
Z2 = A1 @ W2 + b2   # (3,5) @ (5,1) + (1,) → (3,1)
print(f"\nFinal predictions (Z2):")
print(Z2)

---
## 4. Geometric Meaning: Transforming Space

### What does multiplying a vector by a matrix do geometrically?
It **moves** the vector to a new location — it's a **linear transformation**.

The matrix defines how to transform every vector in 2D space. This is how:
- Rotations work (rotate everything by an angle)
- Scaling works (stretch or compress)
- Shearing works (tilt)

This is also why neural networks work: each layer **transforms the data into a new representation** where the patterns are easier to find.

In [ ]:
# ─────────────────────────────────────────────────────────
# GEOMETRIC VIEW: matrices transform shapes
# ─────────────────────────────────────────────────────────

def apply_transform_to_shape(M, ax, title):
    """Draw original unit square and its transformation by M."""
    # Unit square: 4 corners + return to start
    corners = np.array([[0,0],[1,0],[1,1],[0,1],[0,0]]).T  # shape (2, 5)
    transformed = M @ corners
    
    ax.fill(corners[0], corners[1], alpha=0.2, color='gray', label='original')
    ax.plot(corners[0], corners[1], 'gray', lw=1.5, linestyle='--')
    
    ax.fill(transformed[0], transformed[1], alpha=0.35, color='steelblue')
    ax.plot(transformed[0], transformed[1], 'steelblue', lw=2, label='transformed')
    
    # Show where the two basis vectors end up
    e1_new = M @ np.array([1., 0.])
    e2_new = M @ np.array([0., 1.])
    ax.annotate('', xy=e1_new, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='red', lw=2))
    ax.annotate('', xy=e2_new, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='green', lw=2))
    
    det = np.linalg.det(M)
    ax.axhline(0, color='k', lw=0.4); ax.axvline(0, color='k', lw=0.4)
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.2)
    ax.legend(fontsize=8, loc='upper left')
    ax.set_title(f"{title}\nMatrix = {M.tolist()}", fontsize=9)

transforms = [
    (np.array([[2., 0.], [0., 1.]]), "Scale x by 2\n(horizontal stretch)"),
    (np.array([[0.,-1.], [1., 0.]]), "Rotate 90°\n(counter-clockwise)"),
    (np.array([[1., 1.], [0., 1.]]), "Shear\n(tilt rightward)"),
    (np.array([[-1.,0.], [0., 1.]]), "Reflect\n(flip left-right)"),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for (M, title), ax in zip(transforms, axes):
    apply_transform_to_shape(M, ax, title)

plt.suptitle('Matrix × Vector = Transform the Space (geometric view of matrix multiply)',
             fontsize=12)
plt.tight_layout()
plt.show()
print("Red arrow = where [1,0] goes. Green arrow = where [0,1] goes.")
print("These two arrows completely define the transformation.")

---
## 5. Properties You Must Know

### NOT commutative: AB ≠ BA!
This is the biggest difference from regular multiplication.

Analogy: putting on socks then shoes ≠ putting on shoes then socks. Order matters!

### Associative: (AB)C = A(BC) ✓
You can change where you put the parentheses, but not the order.

### Transpose of product reverses order: (AB)ᵀ = BᵀAᵀ
This shows up constantly in backpropagation.

In [ ]:
# ─────────────────────────────────────────────────────────
# PROPERTIES
# ─────────────────────────────────────────────────────────

np.random.seed(0)
A = np.random.randn(3, 3)
B = np.random.randn(3, 3)
C = np.random.randn(3, 3)

print("=== NOT commutative: AB ≠ BA ===")
print(f"A @ B == B @ A? {np.allclose(A@B, B@A)}")
print("(False — order MATTERS in matrix multiply)")
print(f"First element of A@B: {(A@B)[0,0]:.4f}")
print(f"First element of B@A: {(B@A)[0,0]:.4f}")

print()
print("=== Associative: (AB)C = A(BC) ===")
print(f"(A@B)@C == A@(B@C)? {np.allclose((A@B)@C, A@(B@C))}")
print("(True — parentheses don't matter, just order)")

print()
print("=== Transpose reverses order: (AB)ᵀ = BᵀAᵀ ===")
print(f"(A@B).T == B.T @ A.T? {np.allclose((A@B).T, B.T @ A.T)}")
print("(This appears in backpropagation!)")

print()
print("=== Concrete non-commutativity example ===")
Rotate = np.array([[0., -1.], [1., 0.]])   # rotate 90°
Scale  = np.array([[3.,  0.], [0., 1.]])   # scale x by 3
v = np.array([1., 0.])
print(f"Starting vector: {v}")
print(f"Rotate THEN scale: Scale @ (Rotate @ v) = {Scale @ (Rotate @ v)}")
print(f"Scale THEN rotate: Rotate @ (Scale @ v) = {Rotate @ (Scale @ v)}")
print("Different results!")

---
## 6. Speed: Why Loops Are Slow, NumPy Is Fast

In [ ]:
import time

# ─────────────────────────────────────────────────────────
# SPEED COMPARISON: Python loops vs NumPy @
# ─────────────────────────────────────────────────────────

def matmul_python_loops(A, B):
    """Matrix multiply using pure Python loops — for understanding only."""
    m, k = A.shape
    _, n = B.shape
    C = np.zeros((m, n))        # initialize result with zeros
    for i in range(m):          # loop over rows of A
        for j in range(n):      # loop over columns of B
            for l in range(k):  # dot product: sum over shared dimension
                C[i, j] += A[i, l] * B[l, j]
    return C

# Test on a small matrix first — make sure it's correct
A_small = np.array([[1.,2.],[3.,4.]])
B_small = np.array([[5.,6.],[7.,8.]])
result_loop = matmul_python_loops(A_small, B_small)
result_numpy = A_small @ B_small
print("Loop result matches NumPy:", np.allclose(result_loop, result_numpy))

# Now benchmark on a 30×30 matrix (100% Python loops is SLOW)
size = 30
A = np.random.randn(size, size)
B = np.random.randn(size, size)

start = time.time()
C_slow = matmul_python_loops(A, B)
time_loop = time.time() - start

# NumPy on a 200×200 matrix (compare)
size2 = 200
A2 = np.random.randn(size2, size2)
B2 = np.random.randn(size2, size2)

start = time.time()
for _ in range(100):
    C_fast = A2 @ B2
time_numpy = (time.time() - start) / 100

print(f"\nPython loops  ({size}×{size}):  {time_loop*1000:.1f} ms")
print(f"NumPy @       ({size2}×{size2}): {time_numpy*1000:.3f} ms")
print()
print("NumPy calls optimized BLAS libraries written in C/Fortran.")
print("These use vectorized CPU instructions (SIMD) and cache-friendly memory.")
print("For ML at scale, using @ is not optional — it's required!")

---
## Summary

| Concept | Key point |
|---|---|
| Shape rule | (m×n) @ (n×p) → (m×p) — inner dims must match |
| Each element | C[i,j] = dot product of row i from A with col j from B |
| In ML | Every layer: Z = X @ W + b |
| Geometric | Matrix transforms/rotates/scales the space |
| Order matters | AB ≠ BA (unlike regular numbers!) |
| Transpose rule | (AB)ᵀ = BᵀAᵀ |
| Always use | `A @ B`, never loop manually |

**Next: Notebook 04 — Determinants, Inverses & Solving Equations** (how do we find unknown values given a matrix equation?)